<a href="https://colab.research.google.com/github/dklishta/python-ai-Gailunaite-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`cartoons_genre_country_duration.csv`](https://github.com/componavt/python-ai-template/blob/main/data/examples/cartoons_genre_country_duration.sparql) — жанры, страны и продолжительность мультфильмов
- [`cartoons_assessment_reviews.csv`](https://github.com/componavt/python-ai-template/blob/main/data/examples/cartoons_assessment_reviews.sparql) — оценки и рецензии мультфильмов

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [5]:
# 🐱 Шаг 1. Клонируем ваш репозиторий курса в Colab

import os

repo = "python-ai-Gailunaite-Darya"  # ← ИЗМЕНЕНО: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Gailunaite-Darya
    !git clone -q https://github.com/dklishta/python-ai-Gailunaite-Darya.git  # ← ИЗМЕНЕНО: URL вашего репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Gailunaite-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [6]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_teacoffee = pd.read_csv("data/tea_coffee_info.csv")
df_teacoffeefin = pd.read_csv("data/tea_coffee_fin.csv")

print("✅ Загружено строк в df_teacoffee:", len(df_teacoffee))
print("✅ Загружено строк в df_teacoffeefin:", len(df_teacoffeefin))

✅ Загружено строк в df_teacoffee: 289
✅ Загружено строк в df_teacoffeefin: 1555


## 🧹 [2B] Очистка и переименование столбцов

В исходных CSV-файлах есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `film` с URL (ссылкой на объект Wikidata) — **сохраняем его для отладки**, но переименуем в `URL`.
- Столбцы `filmLabel`, `genreLabel`, `countryLabel`, `assessmentLabel`, `outcomeLabel` содержат читаемые подписи (названия).

В этом шаге мы:
- переименуем столбец с URL Wikidata (`film` → `URL`);
- переименуем `filmLabel → film`, `genreLabel → genre`, `countryLabel → country`, `assessmentLabel → assessment`, `outcomeLabel → outcome`;
- приведём числовые столбцы (`duration`, `publicationYear`) к типу `int`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [9]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для tea_coffee_info.csv
# (с защитой от NameError и улучшенной загрузкой данных)

import pandas as pd
import os

# 🔁 Если df не определён — пробуем загрузить файл заново
if 'df' not in globals():
    print("⚠️ Переменная 'df' не найдена. Пробуем загрузить данные...")

    file_path = "data/tea_coffee_info.csv"

    # Проверяем существование файла
    if not os.path.exists(file_path):
        # Пробуем альтернативные пути (частая проблема в Colab)
        alt_paths = [
            "/content/python-ai-Gailunaite-Darya/data/tea_coffee_info.csv",
            "tea_coffee_info.csv",
            "../data/tea_coffee_info.csv"
        ]
        for alt in alt_paths:
            if os.path.exists(alt):
                file_path = alt
                print(f"✅ Файл найден по альтернативному пути: {file_path}")
                break
        else:
            raise FileNotFoundError(
                f"❌ Файл не найден!\n"
                f"Текущая папка: {os.getcwd()}\n"
                f"Ожидаемый путь: data/tea_coffee_info.csv\n"
                f"Содержимое текущей папки: {os.listdir('.')}"
            )

    # Пробуем разные кодировки и разделители
    df = None
    for encoding in ["utf-8", "cp1251", "utf-8-sig"]:
        for sep in [",", ";", "\t"]:
            try:
                df = pd.read_csv(file_path, encoding=encoding, sep=sep)
                print(f"✅ Файл успешно загружен: encoding='{encoding}', sep='{sep}'")
                break
            except Exception:
                continue
        if df is not None:
            break

    if df is None:
        raise ValueError(
            "❌ Не удалось прочитать CSV-файл.\n"
            "Проверьте: кодировку файла, разделитель, целостность данных."
        )

# ============================================================
# 🧹 Основная логика очистки и переименования
# ============================================================

# 1) Удаляем технический столбец с URL Wikidata (если существует)
if "company" in df.columns:
    df = df.drop(columns=["company"])
    print("✅ Столбец 'company' (URL) удалён")
else:
    print("⏭️ Столбец 'company' не найден, пропускаем удаление")

# 2) Переименовываем столбцы: убираем суффикс Label
rename_map = {
    "companyLabel": "company",
    "productTypeLabel": "productType",
    "countryLabel": "country",
    "hqLabel": "hq",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
print("✅ Столбцы переименованы:", list(df.columns))

# 3) Приводим foundingYearShort к числовому типу (если столбец существует)
if "foundingYearShort" in df.columns:
    df["foundingYearShort"] = pd.to_numeric(
        df["foundingYearShort"], errors="coerce"
    ).fillna(0).astype(int)
    print("✅ foundingYearShort приведён к типу int")
else:
    print("⏭️ Столбец foundingYearShort не найден, пропускаем преобразование")

# 4) Показываем результат
print("\n📋 Первые 5 строк после очистки:")
display(df.head())

print(f"\n📊 Информация о DataFrame: {df.shape[0]} строк, {df.shape[1]} столбцов")
print("\n✅ Данные готовы к анализу")

⚠️ Переменная 'df' не найдена. Пробуем загрузить данные...
✅ Файл успешно загружен: encoding='utf-8', sep=','
✅ Столбец 'company' (URL) удалён
✅ Столбцы переименованы: ['company', 'productType', 'country', 'hq', 'hqCoord', 'foundingYearShort']
✅ foundingYearShort приведён к типу int

📋 Первые 5 строк после очистки:


,company,productType,country,hq,hqCoord,foundingYearShort
0,Британская Ост-Индская компания,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
1,Британская Ост-Индская компания,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
2,Тим Хортонс,кофе,Канада,Оквилл,Point(-79.683333333 43.45),1964
3,Nespresso,кофе,Швейцария,Лозанна,Point(6.633333333 46.533333333),1986
4,Kraft Foods,кофе,США,Нортфилд,Point(-87.766666666 42.1),2012



📊 Информация о DataFrame: 289 строк, 6 столбцов

✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по бюджету (`capital_cost`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_genre, "Жанры, страны и продолжительность (df_genre)")
show_info(df_assessment, "Год выпуска, оценки и рейтинги (df_assessment)")


📊 Жанры, страны и продолжительность (df_genre)
Размер: (2596, 6)
Столбцы: URL, film, genre, country, duration, capital_cost

Первые строки:
                                       URL                       film  \
0  http://www.wikidata.org/entity/Q1128756                Мэри и Макс   
1  http://www.wikidata.org/entity/Q1199692  Отважный маленький тостер   
2  http://www.wikidata.org/entity/Q1514402                  Ренессанс   
3  http://www.wikidata.org/entity/Q1514402                  Ренессанс   
4  http://www.wikidata.org/entity/Q1418615     Приключения кота Фрица   

                                    genre     country  duration  capital_cost  
0                       взрослая анимация   Австралия        90     8240000.0  
1  экранизация литературного произведения         США        90     2300000.0  
2                               киберпанк  Люксембург       101    18000000.0  
3                                 неонуар     Франция       101    18000000.0  
4  экранизация литер

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** фильмов, стран, жанров есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу строк);
- **какие жанры самые популярные** (Топ‑10 по числу строк);
- **какие оценки (assessment) и результаты (outcome) присутствуют** в данных об оценках.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_genre["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет 1: жанры, страны, длительность
print("\n📊 Датасет: Жанры, страны и продолжительность")
print("Уникальных мультфильмов в df_genre:", df_genre["film"].nunique())
print("Уникальных стран:", df_genre["country"].nunique())
print("Уникальных жанров:", df_genre["genre"].nunique())

print("\nТоп-5 стран по числу записей:")
print(df_genre["country"].value_counts().head())

print("\nТоп-10 жанров:")
print(df_genre["genre"].value_counts().head(10))

# Датасет 2: оценки и рейтинги
print("\n📊 Датасет: Год выпуска, оценки и рейтинги")
print("Уникальных мультфильмов в df_assessment:", df_assessment["film"].nunique())
print("Уникальных типов оценок:", df_assessment["assessment"].nunique())
print("Диапазон лет:", df_assessment["publicationYear"].min(), "—", df_assessment["publicationYear"].max())

print("\nТипы оценок (assessment):")
print(df_assessment["assessment"].value_counts())

print("\nРезультаты оценок (outcome):")
print(df_assessment["outcome"].value_counts())

print("\nСтатистика по годам публикации:")
print(df_assessment["publicationYear"].describe())

🔍 Быстрая проверка данных

📊 Датасет: Жанры, страны и продолжительность
Уникальных мультфильмов в df_genre: 418
Уникальных стран: 40
Уникальных жанров: 125

Топ-5 стран по числу записей:
country
США               1303
Франция            196
Великобритания     125
Россия              91
Дания               82
Name: count, dtype: int64

Топ-10 жанров:
genre
приключенческий фильм          353
комедийный фильм               305
фэнтезийный фильм              289
семейный фильм                 247
детский фильм                  187
музыкальный фильм              120
драматический фильм             91
боевик                          84
научно-фантастический фильм     74
мультфильм                      70
Name: count, dtype: int64

📊 Датасет: Год выпуска, оценки и рейтинги
Уникальных мультфильмов в df_assessment: 4929
Уникальных типов оценок: 22
Диапазон лет: 0 — 2027

Типы оценок (assessment):
assessment
тест Бекдел                                                                   1426
обрат

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨